<a href="https://colab.research.google.com/github/mahdisahad/WOA-Feature-Selection/blob/main/notebooks/02_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="text-align: right">

# نوت‌بوک ۲: پیش‌پردازش داده (Data Preprocessing)

در این نوت‌بوک داده را برای مدل‌سازی آماده می‌کنیم و ترتیب درست کارها را نشان می‌دهیم:

1. داده را بارگذاری و ویژگی‌ها و برچسب را جدا می‌کنیم.
2. **اول** داده را به train و test تقسیم می‌کنیم.
3. **بعد** ویژگی‌های تکراری را فقط با نگاه به train شناسایی و حذف می‌کنیم.
4. **بعد** استانداردسازی را فقط روی train یاد می‌گیریم (`fit`) و روی test اعمال می‌کنیم (`transform`).

قاعده‌ی اصلی: هر چیزی که از داده «یاد گرفته می‌شود» (ویژگی تکراری، میانگین، انحراف معیار) فقط باید از train یاد گرفته شود. وگرنه اطلاعات test به train نشت می‌کند و نتیجه‌ی نهایی خوش‌بینانه می‌شود (data leakage).

نکته: داده‌ی استانداردشده‌ی این نوت‌بوک (`X_train_scaled`) در نوت‌بوک‌های بعدی استفاده نمی‌شود؛ آن‌ها استانداردسازی را داخل `Pipeline` هر مدل انجام می‌دهند. این نوت‌بوک برای نشان دادن و بررسی مراحل پیش‌پردازش است.

</div>

<div dir="rtl" style="text-align: right">

### اتصال به Google Drive
Drive به Colab وصل می‌شود تا به پوشه‌ی پروژه و دیتاست‌ها دسترسی داشته باشیم.

</div>

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

<div dir="rtl" style="text-align: right">

### بررسی پوشه‌ی پروژه و فایل‌های دیتاست
مسیر پروژه (`PROJECT_PATH`) و پوشه‌ی `datasets` تعریف می‌شود. با `exists()` بررسی می‌کنیم هر دو وجود دارند و سپس نام فایل‌ها و پوشه‌های داخل `datasets` چاپ می‌شود. هدف فقط اطمینان از درست بودن مسیرها قبل از خواندن داده است.

</div>

In [ ]:
from pathlib import Path

PROJECT_PATH = Path("/content/drive/MyDrive/WOA_Project")
DATASETS_PATH = PROJECT_PATH / "datasets"

print("Project folder exists:", PROJECT_PATH.exists())
print("Datasets folder exists:", DATASETS_PATH.exists())

print("\nFiles inside datasets:")

if DATASETS_PATH.exists():
    for file in DATASETS_PATH.iterdir():
        print(file.name)
else:
    print("Datasets folder not found")

<div dir="rtl" style="text-align: right">

### بارگذاری دیتاست فعال
- `sys.path.append(...)` پوشه‌ی پروژه را به مسیر جست‌وجوی پایتون اضافه می‌کند تا `dataset_config.py` پیدا شود.
- `from dataset_config import *` توابع وابسته به دیتاست (`load_dataset`، `split_xy`، `split_train_test`، ...) و `ACTIVE_DATASET` را وارد می‌کند. با عوض کردن همان یک متغیر، کل نوت‌بوک روی دیتاست دیگری اجرا می‌شود.
- `load_dataset` داده را به یک جدول با ستون هدف `class` تبدیل می‌کند و ابعاد و پنج ردیف اول نمایش داده می‌شود.

</div>

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Define project paths
PROJECT_PATH = Path("/content/drive/MyDrive/WOA_Project")
import sys
sys.path.append(str(PROJECT_PATH))
from dataset_config import *   # <-- dataset-specific code lives only here
print("Active dataset:", ACTIVE_DATASET)

# Load dataset
df = load_dataset(PROJECT_PATH)

print("Dataset loaded successfully")
print("Dataset shape:", df.shape)

# Display first five rows
display(df.head())


<div dir="rtl" style="text-align: right">

### نگاهی به ساختار جدول
نام ستون‌ها، توزیع نوع داده‌ها (`dtypes`)، نام آخرین ستون و توزیع کلاس‌ها چاپ می‌شود. چون `load_dataset` ستون `class` را همیشه آخر می‌گذارد، `df.columns[-1]` و `df.iloc[:, -1]` همان ستون هدف است. نتیجه نشان می‌دهد همه‌ی ویژگی‌ها عددی‌اند و کلاس‌ها چقدر متوازن‌اند.

</div>

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes.value_counts())

print("\nLast column name:")
print(df.columns[-1])

print("\nClass distribution:")
print(df.iloc[:, -1].value_counts())


<div dir="rtl" style="text-align: right">

### جدا کردن ویژگی‌ها (X) و برچسب (y)
`split_xy` جدول را به سه بخش تقسیم می‌کند:
- `X`: فقط ستون‌های ویژگی،
- `y`: ستون `class`،
- `groups`: شناسه‌ی فرد (فقط برای Parkinson که هر فرد چند ردیف دارد؛ برای بقیه `None`).

سپس ابعاد و کلاس‌های یکتا چاپ می‌شود تا مطمئن شویم جداسازی درست بوده است.

</div>

In [ ]:
# Separate features and target

X, y, groups = split_xy(df)

# Check dimensions
print("Features shape:", X.shape)
print("Target shape:", y.shape)

# Check class labels
print("\nUnique classes:", y.unique())

# Display first five target values
print("\nFirst five target values:")
print(y.head())


<div dir="rtl" style="text-align: right">

### بررسی مقدار گم‌شده
`X.isnull().sum()` تعداد مقدار خالی هر ستون را می‌دهد. تعداد کل، تعداد ستون‌های دارای مقدار خالی و مقدار خالی در برچسب‌ها چاپ می‌شود. اگر صفر نباشد باید قبل از مدل‌سازی (مثلاً با جایگزینی) رسیدگی شود. این داده‌ها بدون مقدار گم‌شده‌اند.

</div>

In [ ]:
# Check missing values in features
missing_values = X.isnull().sum()

print("Total missing values:", missing_values.sum())
print("Number of features with missing values:",
      (missing_values > 0).sum())

# Check missing values in target
print("Missing values in target:", y.isnull().sum())

<div dir="rtl" style="text-align: right">

### تقسیم داده به train و test
`split_train_test` داده را ۸۰٪ train و ۲۰٪ test تقسیم می‌کند با `random_state=42` برای تکرارپذیری.
- برای دیتاست‌های معمولی از `train_test_split` با `stratify=y` استفاده می‌شود، یعنی نسبت کلاس‌ها در train و test تقریباً برابر با کل داده می‌ماند (در داده‌ی نامتوازن مهم است).
- برای Parkinson تقسیم **بر اساس فرد** انجام می‌شود (`StratifiedGroupKFold`)، چون هر فرد چند ردیف مشابه دارد و اگر ردیف‌های یک فرد هم در train و هم در test باشند، نتیجه‌ی test غیرواقعی بالا می‌رود.

ابعاد و توزیع کلاس هر دو بخش چاپ می‌شود. test از این‌جا به بعد دست نمی‌خورد و فقط برای ارزیابی نهایی است.

</div>

In [ ]:
from sklearn.model_selection import train_test_split

# Split data into training and testing sets
X_train, X_test, y_train, y_test, groups_train = split_train_test(X, y, groups)

# Display shapes
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

# Display class distribution
print("\nTraining class distribution:")
print(y_train.value_counts())

print("\nTesting class distribution:")
print(y_test.value_counts())


<div dir="rtl" style="text-align: right">

### شناسایی ویژگی‌های تکراری (فقط روی train)
`X_train.T.duplicated()` جدول را ترانهاده می‌کند (ویژگی‌ها می‌شوند ردیف) و برای هر ویژگی بررسی می‌کند آیا مقادیرش دقیقاً با ویژگی‌ای قبل‌تر یکی است. از هر گروه تکراری نسخه‌ی اول نگه داشته می‌شود و بقیه علامت می‌خورند. این کار عمداً فقط روی train انجام می‌شود تا تصمیم‌گیری درباره‌ی ویژگی‌ها به test وابسته نباشد. ویژگی‌های تکراری اطلاعات جدیدی ندارند و فقط فضای جست‌وجو را برای الگوریتم نهنگ بزرگ‌تر می‌کنند.

</div>

In [ ]:
# Identify duplicate features using training data only
duplicate_columns = X_train.columns[
    X_train.T.duplicated()
].tolist()

print("Number of duplicate features:", len(duplicate_columns))

print("\nDuplicate feature names:")
print(duplicate_columns)

<div dir="rtl" style="text-align: right">

### حذف ویژگی‌های تکراری از train و test
همان ستون‌هایی که روی train شناسایی شدند از هر دو مجموعه حذف می‌شوند، تا train و test ستون‌های یکسان داشته باشند.

توجه: در این نوت‌بوک فقط ویژگی‌های تکراری حذف می‌شوند. حذف ستون‌های ثابت (`drop_near_constant`) در نوت‌بوک‌های ۰۳ تا ۰۶ انجام می‌شود، برای همین تعداد ویژگی‌ها در این نوت‌بوک ممکن است کمی بیشتر از بقیه‌ی نوت‌بوک‌ها باشد (مثلاً در TCGA یک ستون ثابت باقی می‌ماند).

</div>

In [ ]:
# Remove duplicate features from training and testing data
X_train = X_train.drop(columns=duplicate_columns)
X_test = X_test.drop(columns=duplicate_columns)

print("Training shape after removing duplicates:", X_train.shape)
print("Testing shape after removing duplicates:", X_test.shape)


<div dir="rtl" style="text-align: right">

### استانداردسازی ویژگی‌ها (StandardScaler)
هر ویژگی به شکل $z = (x - \mu) / \sigma$ تبدیل می‌شود تا میانگین صفر و انحراف معیار یک داشته باشد. مدل‌هایی مثل Logistic Regression و SVM به مقیاس ویژگی‌ها حساس‌اند و بدون استانداردسازی ویژگی‌های با مقادیر بزرگ بر بقیه غلبه می‌کنند.

- `fit_transform(X_train)`: میانگین و انحراف معیار را **فقط از train** یاد می‌گیرد و train را تبدیل می‌کند.
- `transform(X_test)`: test را با **همان** میانگین و انحراف معیار train تبدیل می‌کند (بدون یاد گرفتن چیز جدید). اگر scaler روی test هم fit می‌شد، نشت داده رخ می‌داد.
- خروجی scaler آرایه‌ی numpy است؛ آن را دوباره به `DataFrame` با همان نام ستون‌ها و اندیس‌ها برمی‌گردانیم تا نام ویژگی‌ها حفظ شود.

</div>

In [ ]:
from sklearn.preprocessing import StandardScaler

# Create scaler
scaler = StandardScaler()

# Fit scaler only on training data
X_train_scaled = scaler.fit_transform(X_train)

# Apply the same scaler to test data
X_test_scaled = scaler.transform(X_test)

# Convert arrays back to DataFrames
X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=X_test.columns,
    index=X_test.index
)

print("Training data shape:", X_train_scaled.shape)
print("Testing data shape:", X_test_scaled.shape)


<div dir="rtl" style="text-align: right">

### بررسی نتیجه‌ی استانداردسازی
روی train میانگینِ میانگین‌ها باید تقریباً صفر و میانگینِ انحراف معیارها حدود یک باشد (`ddof=0` همان روشی است که `StandardScaler` به کار می‌برد). اگر ستون ثابتی باقی مانده باشد، انحراف معیارش صفر می‌ماند و میانگین کل کمی کمتر از یک می‌شود. کمترین و بیشترین مقدار نشان می‌دهد چقدر داده‌ی پرت (دورتر از چند انحراف معیار) داریم. ابعاد test هم چاپ می‌شود تا مطمئن شویم با train هم‌ستون است.

</div>

In [ ]:
print("Mean of training feature means:",
      X_train_scaled.mean().mean())

print("Mean of training feature standard deviations:",
      X_train_scaled.std(ddof=0).mean())

print("Minimum value in training data:",
      X_train_scaled.min().min())

print("Maximum value in training data:",
      X_train_scaled.max().max())

print("\nTest data shape:", X_test_scaled.shape)
